# 02 - Descripción de los datos

Para cada fuente: qué es una fila, cuántos registros y columnas tiene, qué periodo cubre y qué significa cada columna. Los significados salen de los metadatos oficiales de NYC Open Data (`/api/views/<id>.json`) y, en el caso de pobreza, del diccionario de datos que viene adjunto al dataset en NYC Open Data.

In [1]:
import json
import os
import urllib.request
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

from config import RAW, BRONZE, SILVER, sesion

spark = sesion("02_descripcion")
pd.set_option("display.max_colwidth", 120)
manifiesto = (spark.read.option("header", True).option("sep", "\t").csv(f"{RAW}/manifiesto.tsv")
              .toPandas().astype({"bytes": "int64", "lineas": "int64"}))
FUENTES = dict(zip(manifiesto["fuente"], manifiesto["id"]))
FUENTES

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/27 16:42:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


{'arrestos_historico': '8h9b-rp9u',
 'arrestos_2026': 'uip8-fykc',
 'choques': 'h9gi-nx95',
 'vehiculos': 'bm4k-52h4',
 'pobreza': 'cts7-vksw',
 'sat_2012': 'f9bf-2cp4',
 'puma_2010': 'k2r4-n9ax'}

## 1. Descripción general

In [2]:
GRANULARIDAD = {
    "arrestos_historico": "un arresto realizado por el NYPD",
    "arrestos_2026": "un arresto realizado por el NYPD (año en curso)",
    "choques": "un choque de tránsito reportado por la policía",
    "vehiculos": "un vehículo involucrado en un choque",
    "pobreza": "una persona encuestada (ACS), con su peso poblacional",
    "sat_2012": "un colegio de secundaria",
}
LLAVE = {"arrestos_historico": "ARREST_KEY", "arrestos_2026": "ARREST_KEY", "choques": "COLLISION_ID",
         "vehiculos": "UNIQUE_ID", "pobreza": "SERIALNO + SPORDER", "sat_2012": "DBN"}
COL_FECHA = {"arrestos_historico": "arrest_date", "arrestos_2026": "arrest_date",
             "choques": "crash_date", "vehiculos": "crash_date"}

filas = []
for fuente in GRANULARIDAD:
    df = spark.read.parquet(f"{BRONZE}/{fuente}")
    fila = {"fuente": fuente, "id NYC Open Data": FUENTES[fuente], "una fila es": GRANULARIDAD[fuente],
            "registros": df.count(), "columnas": len(df.columns), "llave": LLAVE[fuente],
            "CSV (MB)": round(manifiesto.set_index("fuente").loc[fuente, "bytes"] / 1024**2, 1)}
    if fuente in COL_FECHA:
        r = df.select(F.to_date(COL_FECHA[fuente], "MM/dd/yyyy").alias("f")).agg(F.min("f"), F.max("f")).first()
        fila["desde"], fila["hasta"] = r[0], r[1]
    filas.append(fila)
general = pd.DataFrame(filas)
general.to_csv("../figuras/tabla_descripcion_general.csv", index=False)
general

,fuente,id NYC Open Data,una fila es,registros,columnas,llave,CSV (MB),desde,hasta
0,arrestos_historico,8h9b-rp9u,un arresto realizado por el NYPD,6264978,19,ARREST_KEY,1279.3,2006-01-01,2025-12-31
1,arrestos_2026,uip8-fykc,un arresto realizado por el NYPD (año en curso),141870,19,ARREST_KEY,24.9,2026-01-01,2026-06-30
2,choques,h9gi-nx95,un choque de tránsito reportado por la policía,2269187,29,COLLISION_ID,454.5,2012-07-01,2026-06-11
3,vehiculos,bm4k-52h4,un vehículo involucrado en un choque,4551002,25,UNIQUE_ID,803.6,2012-07-01,2026-06-11
4,pobreza,cts7-vksw,"una persona encuestada (ACS), con su peso poblacional",68273,61,SERIALNO + SPORDER,13.1,NaN,NaN
5,sat_2012,f9bf-2cp4,un colegio de secundaria,478,6,DBN,0.0,NaN,NaN


Aparte está PUMA 2010: son 55 polígonos, uno por barrio, que se usan en el notebook 05 para ubicar arrestos y choques.

## 2. Metadatos oficiales

Se descargan una sola vez y quedan guardados en `data/raw/metadatos/`.

In [3]:
CARPETA_META = Path("../data/raw/metadatos")
CARPETA_META.mkdir(parents=True, exist_ok=True)
BASE = os.environ.get("NYC_BASE_URL", "https://data.cityofnewyork.us")

def metadatos(ident):
    archivo = CARPETA_META / f"{ident}.json"
    if not archivo.exists():
        try:
            with urllib.request.urlopen(f"{BASE}/api/views/{ident}.json", timeout=60) as r:
                archivo.write_bytes(r.read())
        except Exception as e:
            print(f"  No se pudieron descargar los metadatos de {ident}: {e}")
            return {}
    return json.loads(archivo.read_text())

META = {fuente: metadatos(ident) for fuente, ident in FUENTES.items()}
for fuente, m in META.items():
    if m:
        print(f"{fuente:>20}: {m.get('name')} (publicado por {m.get('attribution')})")

  arrestos_historico: NYPD Arrests Data (Historic) (publicado por Police Department (NYPD))
       arrestos_2026: NYPD Arrest Data (Year to Date) (publicado por Police Department (NYPD))
             choques: Motor Vehicle Collisions - Crashes (publicado por Police Department (NYPD))
           vehiculos: Motor Vehicle Collisions - Vehicles (publicado por Police Department (NYPD))
             pobreza: NYCgov Poverty Measure Data (2018) (publicado por Mayor's Office for Economic Opportunity)
            sat_2012: 2012 SAT Results (publicado por Department of Education (DOE))
           puma_2010: 2010 Public Use Microdata Areas (PUMAs) (publicado por Department of City Planning (DCP))


In [4]:
import re

def normalizar(nombre):
    return re.sub(r"[^0-9a-z]+", "_", nombre.strip().lower()).strip("_")

def perfil(fuente):
    """descripción oficial, % presentes, distintos y ejemplo por columna"""
    df = spark.read.parquet(f"{BRONZE}/{fuente}")
    n = df.count()
    presentes = df.agg(*[F.count(F.when(F.trim(F.col(c)) != "", 1)).alias(c) for c in df.columns]).first().asDict()
    distintos = df.agg(*[F.approx_count_distinct(c).alias(c) for c in df.columns]).first().asDict()
    ejemplo = df.dropna(how="any").first() or df.first()
    oficial = {normalizar(c["name"]): (c.get("dataTypeName", ""), " ".join((c.get("description") or "").split()))
               for c in META.get(fuente, {}).get("columns", [])}
    return pd.DataFrame([{
        "columna": c,
        "tipo en la fuente": oficial.get(c, ("", ""))[0],
        "descripción oficial": oficial.get(c, ("", "(sin descripción oficial)"))[1] or "(sin descripción oficial)",
        "% presentes": round(100 * presentes[c] / n, 1),
        "valores distintos (aprox.)": distintos[c],
        "ejemplo": str(ejemplo[c])[:40] if ejemplo and ejemplo[c] is not None else "",
    } for c in df.columns])

perfiles = {}

### Arrestos
Las dos tablas de arrestos tienen las mismas columnas, así que solo se describe la histórica.

In [5]:
perfiles["arrestos"] = perfil("arrestos_historico"); perfiles["arrestos"]

26/09/27 16:43:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,columna,tipo en la fuente,descripción oficial,% presentes,valores distintos (aprox.),ejemplo
0,arrest_key,text,Randomly generated persistent ID for each arrest,100.0,6080957,147056919
1,arrest_date,calendar_date,Exact date of arrest for the reported event,100.0,7478,10/19/2015
2,pd_cd,number,Three digit internal classification code (more granular than Key Code),100.0,359,792
3,pd_desc,text,Description of internal classification corresponding with PD code (more granular than Offense Description),99.9,444,WEAPONS POSSESSION 1 & 2
4,ky_cd,number,Three digit internal classification code (more general category than PD code),99.8,78,118
5,ofns_desc,text,Description of internal classification corresponding with KY code (more general category than PD description),99.9,89,DANGEROUS WEAPONS
6,law_code,text,"Law code charges corresponding to the NYS Penal Law, VTL and other various local laws",100.0,2934,PL 265031B
7,law_cat_cd,text,"Level of offense: felony, misdemeanor, violation",99.6,6,F
8,arrest_boro,text,"Borough of arrest. B(Bronx), S(Staten Island), K(Brooklyn), M(Manhattan), Q(Queens)",100.0,5,Q
9,arrest_precinct,number,Precinct where the arrest occurred,100.0,80,102


### Choques

In [6]:
perfiles["choques"] = perfil("choques"); perfiles["choques"]

,columna,tipo en la fuente,descripción oficial,% presentes,valores distintos (aprox.),ejemplo
0,crash_date,calendar_date,Occurrence date of collision,100.0,5295,06/14/2013
1,crash_time,text,Occurrence time of collision,100.0,1357,17:40
2,borough,text,Borough where collision occurred,69.5,5,MANHATTAN
3,zip_code,text,Postal code of incident occurrence,69.5,242,10036
4,latitude,number,"Latitude coordinate for Global Coordinate System, WGS 1984, decimal degrees (EPSG 4326)",89.4,133756,40.7642812
5,longitude,number,"Longitude coordinate for Global Coordinate System, WGS 1984, decimal degrees (EPSG 4326)",89.4,112445,-73.9985813
6,location,location,"Latitude , Longitude pair",89.4,357837,"(40.7642812, -73.9985813)"
7,on_street_name,text,Street on which the collision occurred,78.0,23328,WEST SIDE HIGHWAY
8,cross_street_name,text,Nearest cross street to the collision,61.7,26324,WEST 46 STREET
9,off_street_name,text,Street address if known,17.9,286382,


### Vehículos

In [7]:
perfiles["vehiculos"] = perfil("vehiculos"); perfiles["vehiculos"]

,columna,tipo en la fuente,descripción oficial,% presentes,valores distintos (aprox.),ejemplo
0,unique_id,number,Unique record code generated by system. Primary Key.,100.0,4804883,20051142
1,collision_id,number,"Crash identification code. Foreign Key, matches unique_id from the Crash table.",100.0,2554237,4452640
2,crash_date,calendar_date,Occurrence date of collision,100.0,5295,08/29/2021
3,crash_time,text,Occurrence time of collision,100.0,1357,10:12
4,vehicle_id,text,Vehicle identification code assigned by system,100.0,3160097,faf13e39-1bdb-4533-aa8d-2d779f2c449e
5,state_registration,text,State where vehicle is registered.,91.9,86,NJ
6,vehicle_type,text,"Type of vehicle based on the selected vehicle category (ATV, bicycle, car/suv, ebike, escooter, truck/bus, motorcycl...",93.8,3136,
7,vehicle_make,text,Vehicle make,57.0,14514,
8,vehicle_model,text,Vehicle model,1.3,4245,
9,vehicle_year,text,Year the vehicle was manufactured,56.5,330,


### Pobreza
En NYC Open Data este dataset no trae descripción de las columnas; lo que significa cada una está en el diccionario de datos adjunto (un Excel), y los códigos que usamos quedaron en `notebooks/codigos.py`.

In [8]:
perfiles["pobreza"] = perfil("pobreza"); perfiles["pobreza"]

26/09/27 16:44:13 WARN DAGScheduler: Broadcasting large task binary with size 1477.4 KiB


,columna,tipo en la fuente,descripción oficial,% presentes,valores distintos (aprox.),ejemplo
0,serialno,number,(sin descripción oficial),100.0,25343,539972
1,sporder,number,(sin descripción oficial),100.0,20,3
2,pwgtp,number,(sin descripción oficial),100.0,722,87
3,wgtp,number,(sin descripción oficial),100.0,561,60
4,agep,number,(sin descripción oficial),100.0,95,27
...,...,...,...,...,...,...
56,semp_adj,number,(sin descripción oficial),100.0,375,0
57,ssip_adj,number,(sin descripción oficial),100.0,157,0
58,ssp_adj,number,(sin descripción oficial),100.0,355,0
59,totalworkhrs_pu,number,(sin descripción oficial),100.0,5,1


### SAT 2012

In [9]:
perfiles["sat"] = perfil("sat_2012"); perfiles["sat"]

,columna,tipo en la fuente,descripción oficial,% presentes,valores distintos (aprox.),ejemplo
0,dbn,text,(sin descripción oficial),100.0,492,25Q252
1,school_name,text,(sin descripción oficial),100.0,476,"QUEENS SCHOOL OF INQUIRY, THE"
2,num_of_sat_test_takers,text,(sin descripción oficial),100.0,175,58
3,sat_critical_reading_avg_score,text,(sin descripción oficial),100.0,158,455
4,sat_math_avg_score,text,(sin descripción oficial),100.0,169,498
5,sat_writing_avg_score,text,(sin descripción oficial),100.0,166,443


In [10]:
pd.concat([p.assign(tabla=t) for t, p in perfiles.items()]).to_csv("../figuras/tabla_atributos.csv", index=False)
print("Diccionario de atributos guardado en figuras/tabla_atributos.csv")

Diccionario de atributos guardado en figuras/tabla_atributos.csv


## 3. Tipos de datos

En bronce todo es texto. Acá se ven los tipos que quedaron en plata.

In [11]:
for tabla in ["arrestos", "choques", "vehiculos", "pobreza", "sat"]:
    df = spark.read.parquet(f"{SILVER}/{tabla}")
    print(f"\n===== {tabla}: {df.count():,} registros, {len(df.columns)} columnas")
    df.printSchema()


===== arrestos: 1,608,509 registros, 24 columnas
root
 |-- arrest_key: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- pd_cd: integer (nullable = true)
 |-- pd_desc: string (nullable = true)
 |-- ky_cd: integer (nullable = true)
 |-- ofns_desc: string (nullable = true)
 |-- law_code: string (nullable = true)
 |-- law_cat_cd: string (nullable = true)
 |-- gravedad: string (nullable = true)
 |-- boro_codigo: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- precinto: integer (nullable = true)
 |-- jurisdiccion: integer (nullable = true)
 |-- age_group_original: string (nullable = true)
 |-- grupo_edad: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raza: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- puma: string (nullable = true)
 |-- fuente: string (nullable = true)
 |-- anio: integer (nullable = true)


===== choques: 846,420 registros, 33 columnas
root
 |-- collision_id: long (nullable = true)
 |-- fecha: date (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- hora: integer (nullable = true)
 |-- minuto: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- fuente_borough: string (nullable = true)
 |-- borough_original: string (nullable = true)
 |-- puma: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- codigo_postal: string (nullable = true)
 |-- heridos: integer (nullable = true)
 |-- muertos: integer (nullable = true)
 |-- grave: boolean (nullable = true)
 |-- number_of_pedestrians_injured: integer (nullable = true)
 |-- number_of_pedestrians_killed: integer (nullable = true)
 |-- number_of_cyclist_injured: integer (nullable = true)
 |-- number_of_cyclist_killed: integer (nullable = true)
 |-- number_of_motorist_injured: integer (nullable = true)
 |-- number_of_mot


===== vehiculos: 1,710,836 registros, 15 columnas
root
 |-- unique_id: long (nullable = true)
 |-- collision_id: long (nullable = true)
 |-- fecha: date (nullable = true)
 |-- tipo_vehiculo: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- anio_vehiculo: integer (nullable = true)
 |-- ocupantes: integer (nullable = true)
 |-- sexo_conductor: string (nullable = true)
 |-- licencia: string (nullable = true)
 |-- maniobra_previa: string (nullable = true)
 |-- punto_impacto: string (nullable = true)
 |-- factor_1: string (nullable = true)
 |-- factor_2: string (nullable = true)
 |-- tiene_choque: boolean (nullable = true)
 |-- anio: integer (nullable = true)




===== pobreza: 68,273 registros, 16 columnas
root
 |-- hogar: long (nullable = true)
 |-- persona: integer (nullable = true)
 |-- peso_persona: integer (nullable = true)
 |-- peso_hogar: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- grupo_edad: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- educacion: string (nullable = true)
 |-- educacion_codigo: integer (nullable = true)
 |-- etnia: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- en_pobreza: boolean (nullable = true)
 |-- estado_pobreza: string (nullable = true)
 |-- en_pobreza_oficial: boolean (nullable = true)
 |-- ingreso_nycgov: double (nullable = true)
 |-- umbral_nycgov: double (nullable = true)




===== sat: 478 registros, 9 columnas
root
 |-- dbn: string (nullable = true)
 |-- colegio: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- estudiantes: integer (nullable = true)
 |-- lectura: integer (nullable = true)
 |-- matematicas: integer (nullable = true)
 |-- escritura: integer (nullable = true)
 |-- suprimido: boolean (nullable = true)
 |-- total: integer (nullable = true)



## 4. Estadísticos básicos (plata)

In [12]:
choques = spark.read.parquet(f"{SILVER}/choques")
choques.select("heridos", "muertos", "number_of_pedestrians_injured", "number_of_cyclist_injured", "hora").summary(
    "count", "mean", "stddev", "min", "50%", "max").toPandas().set_index("summary")

,heridos,muertos,number_of_pedestrians_injured,number_of_cyclist_injured,hora
summary,,,,,
count,846419,839861,846420,846420,846420
mean,0.4569096393157526,0.0023158594100690473,0.07701141277380025,0.04488079204177595,12.895675905578791
stddev,0.7948378502486224,0.05055470236662817,0.2806929025446252,0.2107973436302355,6.1153707356843015
min,0,0,0,0,0
50%,0,0,0,0,14
max,40,5,19,3,23


In [13]:
vehiculos = spark.read.parquet(f"{SILVER}/vehiculos")
vehiculos.select("anio_vehiculo", "ocupantes").summary("count", "mean", "stddev", "min", "50%", "max").toPandas().set_index("summary")

,anio_vehiculo,ocupantes
summary,,
count,1392269,1506179
mean,2014.408105761171,1145.3674490216633
stddev,6.107130075295241,896775.9734049656
min,1920,0
50%,2016,1
max,2027,981990849


In [14]:
sat = spark.read.parquet(f"{SILVER}/sat")
sat.select("estudiantes", "lectura", "matematicas", "escritura", "total").summary(
    "count", "mean", "stddev", "min", "50%", "max").toPandas().set_index("summary")

,estudiantes,lectura,matematicas,escritura,total
summary,,,,,
count,421,421,421,421,421
mean,110.3206650831354,400.8503562945368,413.3681710213777,393.9857482185273,1208.2042755344419
stddev,155.5342536933185,56.802782721207876,64.68465508566645,58.63510882465652,174.53612282892178
min,6,279,312,286,887
50%,62,391,395,381,1169
max,1277,679,735,682,2096


In [15]:
# registros por año
por_anio = None
for tabla in ["arrestos", "choques", "vehiculos"]:
    t = spark.read.parquet(f"{SILVER}/{tabla}").groupBy("anio").count().withColumnRenamed("count", tabla)
    por_anio = t if por_anio is None else por_anio.join(t, "anio", "outer")
por_anio = por_anio.orderBy("anio").toPandas().set_index("anio")
por_anio.to_csv("../figuras/tabla_registros_por_anio.csv")
por_anio

,arrestos,choques,vehiculos
anio,,,
2019,214617,211486,426724
2020,140413,112918,231211
2021,155507,110558,227267
2022,189774,103887,210614
2023,226872,96607,194967
2024,260503,91316,182839
2025,278953,85546,170015
2026,141870,34102,67199


Ojo que 2026 solo tiene una parte del año, entonces no se puede comparar directo con los otros.

## Conclusiones

- Hay 6 fuentes en forma de tabla y 1 geográfica (las PUMAs). Las tres más grandes tienen millones de filas.
- Cada fuente tiene su propio nivel de detalle: arresto, choque, vehículo, persona o colegio.
- En bronce todo es texto; los tipos se asignan en plata.